# 00 · Setup & Your First Claude Call  🟢 Beginner

**Goal:** make sure your environment works and understand the building block every LangGraph app uses: a *chat model*.

| Level | Notebook | You will learn |
|---|---|---|
| 🟢 Beginner | **00** Setup & first LLM call | API key, `ChatAnthropic`, messages, streaming |
| 🟢 Beginner | 01 Graph basics | State, nodes, edges, reducers |
| 🟢 Beginner | 02 Conditional edges & loops | Routing, cycles, recursion limit, `Command` |
| 🟢 Beginner | 03 Chatbot with an LLM | `MessagesState`, `add_messages`, prompt chaining |
| 🟡 Intermediate | 04 Tools & ReAct agent | Tool calling, `ToolNode`, `create_agent` |
| 🟡 Intermediate | 05 Memory & persistence | Checkpointers, threads, SQLite, trimming |
| 🟡 Intermediate | 06 Human-in-the-loop | `interrupt()`, approve / edit / reject |
| 🟡 Intermediate | 07 Streaming | values / updates / messages / custom streams |
| 🟠 Advanced | 08 Structured output, routing, parallelism | Pydantic outputs, fan-out, `Send` map-reduce |
| 🟠 Advanced | 09 Subgraphs | Composing graphs, private state |
| 🟠 Advanced | 10 Multi-agent supervisor | Handoffs with `Command`, specialist agents |
| 🔴 Expert | 11 Long-term memory (Store) | Cross-thread memory, runtime context |
| 🔴 Expert | 12 Production patterns | Retries, caching, time travel, durability, async |
| 🔴 Expert | 13 Capstone: support agent | Everything together |

## 1. Install packages (run once)

If you created the `.venv` from `requirements.txt` you can skip this. Otherwise uncomment and run:

In [1]:
# %pip install -r ../requirements.txt

## 2. Load the API key from `.env`

Your `.env` (in the project root) contains `CLAUDE_KEY=sk-ant-...`.
`python-dotenv` loads it into environment variables. **Never print or commit the key.**

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # searches upward from this folder for .env

key = os.getenv("CLAUDE_KEY")
print("CLAUDE_KEY loaded:", bool(key), "| starts with:", key[:7] + "..." if key else None)

CLAUDE_KEY loaded: True | starts with: sk-ant-...


## 3. Create the chat model

LangGraph is model-agnostic; it talks to Claude through LangChain's `ChatAnthropic` wrapper.
We pass our key explicitly because it lives in `CLAUDE_KEY` (the library's default name would be `ANTHROPIC_API_KEY`).

In [3]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-opus-5-5",   # Claude Opus 5.5
    api_key=key,
    max_tokens=4096,           # cap on thinking + answer tokens
)

response = llm.invoke("In one sentence: what is LangGraph?")
print(response.text)

LangGraph is an open-source framework from the LangChain team for building stateful, multi-step LLM applications and agents by modeling their workflows as graphs of nodes and edges, with support for cycles, persistence, and human-in-the-loop control.


### What came back?

`invoke` returns an `AIMessage`. Claude Opus 5.5 **thinks before answering**, so `response.content` can be a
*list of blocks* (`thinking` + `text`). Use **`response.text`** to get just the answer.

In [4]:
print(type(response).__name__)
print("block types:", [b["type"] for b in response.content] if isinstance(response.content, list) else "plain string")
print("token usage:", response.usage_metadata)

AIMessage
block types: ['thinking', 'text']
token usage: {'input_tokens': 22, 'output_tokens': 155, 'total_tokens': 177, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_token_details': {'reasoning': 74}}


## 4. Messages: the language of chat models

A conversation is a **list of messages**: `SystemMessage` (instructions), `HumanMessage` (user), `AIMessage` (model).
This list is exactly what LangGraph will later store in *state*.

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a friendly tutor. Answer in at most 2 sentences."),
    HumanMessage("Hi, I'm Ravi."),
    AIMessage("Hi Ravi! What would you like to learn today?"),
    HumanMessage("What's my name, and what is a graph in computer science?"),
]
print(llm.invoke(messages).text)

Your name is Ravi! In computer science, a graph is a data structure made of nodes (vertices) connected by edges, used to model relationships like social networks, road maps, or web links.


👉 The model has **no memory**. It only "remembered" the name because we sent the whole history.
Managing that history (state) is one of the main jobs of LangGraph.

## 5. Controlling effort

`effort` trades depth of thinking for speed/cost: `low`, `medium` (Opus 5.5 default), `high`, `xhigh`, `max`.

In [6]:
fast_llm = ChatAnthropic(model="claude-opus-5-5", api_key=key, max_tokens=4096,
                         output_config={"effort": "low"})
print(fast_llm.invoke("Give me 3 fruit names, comma separated.").text)

Apple, banana, mango


## 6. Streaming tokens

For chat UIs you want tokens as they are generated:

In [7]:
for chunk in llm.stream("Write a 4-line poem about graphs and agents."):
    print(chunk.text, end="", flush=True)

**

Nodes and

 W

anderers**

Across

 the l

...

attered points

 become

 a map that

 min

ds can understand.

## 7. The `helpers.py` shortcut used in every next notebook

To keep the following notebooks focused on LangGraph, `notebooks/helpers.py` wraps the setup above:

* `get_llm(effort=None)` → a `ChatAnthropic` using `CLAUDE_KEY` (model can be overridden with `CLAUDE_MODEL=...` in `.env`).
  It also enables `prefix_mismatch_behavior: "drop_block"`: Claude's thinking blocks are tied to the exact conversation
  that produced them, and LangGraph apps often reshape history (trimming, multi-agent). This setting makes the API drop stale
  thinking blocks instead of rejecting the request.
* `show_graph(graph)` → draws a compiled graph.
* `print_messages(msgs)` → readable transcript (who said what, which tools were called).

In [8]:
from helpers import get_llm, show_graph, print_messages

llm = get_llm()   # ChatAnthropic -> Claude, key read from CLAUDE_KEY in .env

In [9]:
print(llm.invoke("Reply with just: setup works ✅").text)

setup works ✅


## ✅ Recap
* Chat models take a **list of messages** and return an `AIMessage`; use `.text` for the answer.
* They are **stateless** → something must hold history, decide what runs next, call tools, pause for humans…
* That "something" is **LangGraph**. Next: build your first graph (no LLM needed!).